# 🧠 AlphaFin: Multi-Agent LangGraph Workflow
## Capstone Project 2 — Use Case A: Financial Market Intelligence

This notebook translates and visualizes the complete **LangGraph multi-agent architecture** implemented in `src/graph/workflow.py`.

### Core Multi-Agent Flow:
1. **Planner Agent**: Classifies user query intent (`market_info`, `analysis`, `portfolio`, `risk`).
2. **Retriever Agent**: Queries the hybrid RAG store (FAISS MMR + BM25 keyword search).
3. **Specialist Agents**:
   - **Analysis Agent**: Synthesizes market insights and corporate earnings.
   - **Portfolio Agent**: Tailors asset allocations based on risk appetite.
   - **Risk Assessment Agent**: Evaluates portfolio beta, sector concentration, and downside risk.
4. **Validator Agent**: Audits factual grounding and initiates retry loops (up to 3 attempts) if quality thresholds are unmet.

--- 
### Step 1: Environment Setup & Imports

In [2]:
import os
import sys

# Ensure project root is in sys.path
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))

from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

print('✅ Environment and LangGraph dependencies loaded successfully!')

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

Error: Failed to create kernel session on nbctl: nbctl vsessions unavailable: request to http://localhost:8888/api/internal/vsessions failed, reason: 

--- 
### Step 2: AgentState Schema Definition (`TypedDict`)
The shared state object passed across all agent nodes during graph traversal.

In [ ]:
from typing import TypedDict, List, Dict, Any

class AgentState(TypedDict):
    query: str
    session_id: str
    risk_appetite: str
    intent: str
    retrieved_docs: List[str]
    sources: List[Dict[str, Any]]
    retriever_output: str
    analysis_output: str
    portfolio_output: str
    risk_output: str
    agent_answer: str
    final_answer: str
    evaluator_verdict: str
    evaluator_score: int
    evaluator_feedback: str
    confidence: float
    attempts: int
    agents_invoked: List[str]
    latency: float
    error: str

def get_initial_state(query: str, session_id: str = 'fin-session-001', risk_appetite: str = 'moderate') -> AgentState:
    return AgentState(
        query=query,
        session_id=session_id,
        risk_appetite=risk_appetite,
        intent='',
        retrieved_docs=[],
        sources=[],
        retriever_output='',
        analysis_output='',
        portfolio_output='',
        risk_output='',
        agent_answer='',
        final_answer='',
        evaluator_verdict='',
        evaluator_score=0,
        evaluator_feedback='',
        confidence=0.0,
        attempts=0,
        agents_invoked=[],
        latency=0.0,
        error='',
    )

print('✅ AgentState TypedDict registered.')

--- 
### Step 3: Specialist Agent Nodes
Importing the modular agent definitions from `src/agents/`.

In [ ]:
from src.agents.planner import planner
from src.agents.retriever_agent import retriever_agent
from src.agents.analysis_agent import analysis_agent
from src.agents.portfolio_agent import portfolio_agent
from src.agents.risk_agent import risk_agent
from src.agents.validator import validator

print('✅ All 6 specialist agents loaded:')
print('  • Planner:        ', planner.__name__)
print('  • Retriever:      ', retriever_agent.__name__)
print('  • Analysis:       ', analysis_agent.__name__)
print('  • Portfolio:      ', portfolio_agent.__name__)
print('  • Risk:           ', risk_agent.__name__)
print('  • Validator:      ', validator.__name__)

--- 
### Step 4: Conditional Routing Edges

In [ ]:
def route_after_planner(state: AgentState) -> str:
    """Planner routes query to retrieval first to gather relevant financial evidence."""
    return 'retriever'

def route_after_retriever(state: AgentState) -> str:
    """Retriever hands off to the appropriate specialist agent based on intent."""
    intent = state.get('intent', 'market_info')
    route_map = {
        'market_info': 'validator',
        'analysis': 'analysis_agent',
        'portfolio': 'portfolio_agent',
        'risk': 'risk_agent',
    }
    return route_map.get(intent, 'validator')

def route_after_validator(state: AgentState) -> str:
    """Validator audits grounding. Retries up to 3 times if needed, else completes."""
    verdict = state.get('evaluator_verdict', 'approved')
    if verdict == 'needs_retry':
        return 'retriever'
    return 'end'

print('✅ Routing logic configured.')

--- 
### Step 5: Build & Compile the StateGraph Workflow

In [ ]:
def build_graph() -> StateGraph:
    graph = StateGraph(AgentState)

    # Register Nodes
    graph.add_node('planner', planner)
    graph.add_node('retriever', retriever_agent)
    graph.add_node('analysis_agent', analysis_agent)
    graph.add_node('portfolio_agent', portfolio_agent)
    graph.add_node('risk_agent', risk_agent)
    graph.add_node('validator', validator)

    # Edges
    graph.add_edge(START, 'planner')
    graph.add_conditional_edges(
        'planner',
        route_after_planner,
        {'retriever': 'retriever'}
    )
    graph.add_conditional_edges(
        'retriever',
        route_after_retriever,
        {
            'validator': 'validator',
            'analysis_agent': 'analysis_agent',
            'portfolio_agent': 'portfolio_agent',
            'risk_agent': 'risk_agent',
        }
    )
    graph.add_edge('analysis_agent', 'validator')
    graph.add_edge('portfolio_agent', 'validator')
    graph.add_edge('risk_agent', 'validator')
    graph.add_conditional_edges(
        'validator',
        route_after_validator,
        {
            'retriever': 'retriever',
            'end': END,
        }
    )
    return graph

# Compile graph with thread-level MemorySaver
graph = build_graph()
checkpointer = MemorySaver()
workflow_app = graph.compile(checkpointer=checkpointer)
print('🎉 Multi-agent workflow graph compiled successfully!')

--- 
### Step 6: Visualizing the Workflow Graph

Let's display both the **Visual Diagram** and the **Terminal ASCII Tree**.

In [ ]:
# Render Visual PNG Diagram
try:
    display(Image(workflow_app.get_graph().draw_mermaid_png()))
except Exception as e:
    print('Could not render PNG directly (network or graphviz dependency):', e)
    print('\nMermaid Chart Code:')
    print(workflow_app.get_graph().draw_mermaid())

In [ ]:
# Render ASCII Graph
print(workflow_app.get_graph().draw_ascii())

--- 
### Step 7: Sample Workflow Execution Trace
Let's test executing a query through the graph to observe multi-agent coordination.

In [ ]:
test_query = "Suggest an asset allocation and evaluate risk for an aggressive investor with ₹10,00,000 capital"
init_state = get_initial_state(query=test_query, risk_appetite='aggressive')

print(f"Executing query: '{test_query}'")
config = {"configurable": {"thread_id": "notebook-demo-thread"}}

try:
    final_result = workflow_app.invoke(init_state, config=config)
    print("\n--- [Execution Output] ---")
    print("Intent Detected:  ", final_result.get('intent'))
    print("Agents Invoked:   ", final_result.get('agents_invoked'))
    print("Confidence Score: ", final_result.get('confidence'))
    print("Evaluator Verdict:", final_result.get('evaluator_verdict'))
    print("\nFinal Response:\n", final_result.get('final_answer') or final_result.get('agent_answer'))
except Exception as err:
    print(f"Invocation note (Requires live Gemini API key configured): {err}")